# Refinamento — Selo Ambiental 2025 (SEMARH-PI)

Transforma o dado **bruto** da edição 2025 do Selo Ambiental num dataset **pronto para o
painel**. Mesma estrutura do notebook da edição 2026 (`refinamento_selo_ambiental.ipynb`),
com as diferenças que a fonte de 2025 traz — listadas abaixo.

| | |
|---|---|
| **Origem** | `s3a://entrada/sermarh_painel/selo_ambiental_2025.parquet` (o mesmo conteúdo do CSV `db_semarh - selo_ambiental_2025.csv`) |
| **Destino** | `nessie.refinamento.semarh_painel_2025` (resultado final) · `..._2025_fases` (as 3 fases) · dois resumos |
| **Grão** | principal: 1 linha por município (os 224 do Piauí) · fases: 1 linha por município × fase (672) |

O contrato de saída é **idêntico** ao de 2026 (mesmas colunas, mesmos nomes, mesmos
valores de `resultado`/`situacao`), então o mesmo código do painel serve as duas abas.

## O que muda em relação a 2026

- **Existem municípios não postulados.** São 12, e na fonte eles vêm com `RESULTADO n`
  = `NÃO POSTULADO`, sem processo, sem habilitação e sem pontuação. Em 2026 todos os 224
  postularam, então esse caso nunca aparecia — aqui ele precisa virar um `resultado`
  próprio, e não `NULL`.
- **Município vem em CAIXA ALTA** (`SÃO GONÇALO DO PIAUÍ`). A edição 2026 grava em caixa
  mista. Normalizamos para caixa mista com as regras do português (conectivos em
  minúscula, algarismos romanos em maiúscula, letra depois do apóstrofo em maiúscula) —
  222 dos 224 nomes passam a bater exatamente com os de 2026. Os 2 restantes divergem na
  própria fonte (`Nazária` × `Nazária do Piauí`, `São Luis` × `São Luís`), não na caixa;
  por isso o join entre edições é sempre por `cod_ibge`, nunca por nome.
- **A fonte traz o auditor de cada fase** (`AUDITOR ANALISTA 1/2`, `AUDITOR 3`), que 2026
  não tem. Vira a coluna `auditor` — preenchida em 2025, ausente em 2026.
- **`habilitado` pode ser vazio na fonte** (os não postulados). Vazio é `false`
  (não habilitado), não `NULL`.

## O que se mantém de 2026

- `cod_ibge` (vem de `aux4`): código IBGE do município sem o prefixo do estado —
  o código completo é `2200000 + cod_ibge`. É a chave estável para join.
- **Município sem habilitação fica sem apuração** (`NULL`), não com `0`. A fonte grava `0`
  em `CRITÉRIOS n` para inabilitados e não postulados — misturar isso com "atendeu zero
  critérios" infla o primeiro balde do gráfico de critérios.
- **Fases 1/2/3** são as fases de avaliação/recurso da edição. A **fase 3 é o resultado
  final** e alimenta a tabela principal; as três vão para `..._2025_fases`.

## 1. Ler o dado bruto (camada de entrada, no MinIO)

In [ ]:
from pyspark.sql import functions as F
from lakehouse import sessao, ler_arquivo, gravar, perfil, listar

spark = sessao("refinamento-selo-ambiental-2025")

# ler_arquivo usa a zona de entrada por padrao -> s3a://entrada/<caminho>
bruto = ler_arquivo(spark, "sermarh_painel/selo_ambiental_2025.parquet")
print("linhas brutas:", bruto.count())
bruto.printSchema()

## 2. Normalizar o nome do município

A fonte de 2025 grava em CAIXA ALTA. Passar para caixa mista não é cosmético: o nome
aparece em filtro, mapa, tabela e CSV exportado, e é assim que ele fica igual ao da aba
de 2026 — o que permite comparar as duas edições lado a lado sem tratar texto na leitura.

`initcap` do Spark não resolve sozinho: capitalizaria `Do`, `Ii` e `D'alcântara`. As três
regras abaixo cobrem os 224 nomes do Piauí.

In [ ]:
import re

from pyspark.sql.types import StringType

# Conectivos ficam em minuscula, exceto quando abrem o nome.
CONECTIVOS = {"de", "do", "da", "dos", "das", "no", "na", "nos", "nas", "e"}
# "PEDRO II", "PIO IX" — algarismo romano fica inteiro em maiuscula. Exige 2+
# letras para nao pegar palavra de uma letra so.
ROMANO = re.compile(r"^[ivx]{2,}$", re.IGNORECASE)


def _titulo(nome):
    """`SÃO GONÇALO DO PIAUÍ` -> `São Gonçalo do Piauí`."""
    if nome is None:
        return None
    palavras = []
    for posicao, palavra in enumerate(nome.split()):
        base = palavra.lower()
        if posicao and base in CONECTIVOS:
            palavras.append(base)
        elif ROMANO.match(palavra):
            palavras.append(palavra.upper())
        else:
            # Maiuscula no inicio e tambem depois do apostrofo:
            # d'alcântara -> D'Alcântara
            palavras.append(re.sub(r"(^|')(\w)", lambda m: m.group(1) + m.group(2).upper(), base))
    return " ".join(palavras)


titulo = F.udf(_titulo, StringType())

# Confere a regra nos nomes reais antes de usa-la em qualquer lugar.
amostra = ["SÃO GONÇALO DO PIAUÍ", "PEDRO II", "PIO IX", "BARRA D'ALCÂNTARA",
           "MORRO CABEÇA NO TEMPO", "OLHO D'ÁGUA DO PIAUÍ"]
for nome in amostra:
    print(f"  {nome:<28} -> {_titulo(nome)}")

## 3. Refinar

Renomeia para nomes limpos (sem acento/espaço), tipa, deriva `selo`, `situacao`,
`pacto_ambiental` e extrai `latitude`/`longitude` do campo de coordenadas.

A derivação vira uma função `refinar(fase)` — as três fases usam exatamente a mesma
regra, mudando só o sufixo das colunas de origem. Daí saem os dois destinos: `ref`
(fase 3, resultado final) e `fases` (as três, formato longo).

In [ ]:
from functools import reduce

from pyspark.sql import DataFrame

# aux3 = "lat, lon"  ->  remove espacos e divide na virgula
coord = F.split(F.regexp_replace(F.col("aux3"), " ", ""), ",")

# O auditor tem nome de coluna diferente por fase na fonte de 2025.
COLUNA_AUDITOR = {1: "AUDITOR ANALISTA 1", 2: "AUDITOR ANALISTA 2", 3: "AUDITOR 3"}


def refinar(fase: int) -> DataFrame:
    """Refina UMA fase. A derivacao e identica nas tres; muda so o sufixo."""
    res = F.upper(F.trim(F.col(f"`RESULTADO {fase}`")))
    # Vazio (nao postulado) e "NAO" contam igual: nao habilitado. Sem o coalesce
    # a comparacao com NULL devolveria NULL, e `habilitado` viraria um booleano
    # de tres estados que nada mais adiante sabe ler.
    habilitado = F.coalesce(F.upper(F.trim(F.col(f"`HABILITADO {fase}`"))) == "SIM", F.lit(False))
    # Quem nao foi habilitado nao tem apuracao. A fonte grava 0 em `CRITÉRIOS n`
    # para inabilitados e nao postulados: "atendeu zero criterios" e "nao foi
    # avaliado" sao coisas diferentes. Normaliza para NULL, senao o painel soma
    # os dois no mesmo balde do grafico de criterios.
    pontos = F.col(f"`PONTOS {fase}`").cast("double")
    criterios = F.col(f"`CRITÉRIOS {fase}`").cast("int")
    auditor = F.trim(F.col(f"`{COLUNA_AUDITOR[fase]}`"))
    return (
        bruto.select(
            titulo(F.trim(F.col("`MUNICÍPIO`"))).alias("municipio"),
            F.trim(F.col("PROCESSO")).alias("processo"),
            # aux4 e o codigo IBGE do municipio sem o prefixo do estado:
            # codigo completo = 2200000 + cod_ibge. Chave estavel para join —
            # e a unica que serve para cruzar 2025 com 2026, porque o nome
            # diverge na fonte em dois municipios.
            F.col("aux4").cast("int").alias("cod_ibge"),
            F.trim(F.col("aux2")).alias("territorio_desenvolvimento"),
            habilitado.alias("habilitado"),
            F.when(habilitado, pontos).alias("pontos"),
            F.when(habilitado, criterios).alias("criterios_atendidos"),
            F.when(auditor != "", auditor).alias("auditor"),
            res.alias("_res"),
            (F.lower(F.trim(F.col("pactos"))) == "sim").alias("pacto_ambiental"),
            coord.getItem(0).cast("double").alias("latitude"),
            coord.getItem(1).cast("double").alias("longitude"),
        )
        .withColumn("resultado",
            F.when(F.col("_res") == "SELO A", "Selo A")
             .when(F.col("_res") == "SELO B", "Selo B")
             .when(F.col("_res") == "SELO C", "Selo C")
             .when(F.col("_res") == "NÃO ELEGÍVEL", "Não elegível")
             .when(F.col("_res") == "NÃO HABILITADO", "Não habilitado")
             # So existe em 2025: 12 municipios nao entraram na edicao.
             .when(F.col("_res") == "NÃO POSTULADO", "Não postulado"))
        .withColumn("selo", F.regexp_extract(F.col("_res"), r"SELO ([ABC])", 1))
        .withColumn("selo", F.when(F.col("selo") == "", None).otherwise(F.col("selo")))
        .withColumn("tem_selo", F.col("_res").startswith("SELO"))
        .withColumn("situacao",
            F.when(F.col("_res").startswith("SELO"), "Com selo")
             .when(F.col("_res") == "NÃO ELEGÍVEL", "Não elegível")
             .when(F.col("_res") == "NÃO HABILITADO", "Não habilitado")
             .otherwise("Não postulado"))
        .drop("_res")
        .select(
            "municipio", "processo", "cod_ibge", "territorio_desenvolvimento",
            "situacao", "resultado", "selo", "tem_selo", "pontos",
            "criterios_atendidos", "habilitado", "auditor", "pacto_ambiental",
            "latitude", "longitude",
        )
    )


# Tabela principal: o resultado FINAL (fase 3). Grao: 1 linha por municipio.
ref = refinar(3)

# Historico das tres fases. Grao: 1 linha por municipio x fase (224 x 3).
# E o que permite ver a evolucao entre as fases sem reprocessar nada.
fases = reduce(
    DataFrame.unionByName,
    [refinar(n).withColumn("fase", F.lit(n)) for n in (1, 2, 3)],
).select("fase", "municipio", "processo", "cod_ibge", "territorio_desenvolvimento",
         "situacao", "resultado", "selo", "tem_selo", "pontos", "criterios_atendidos",
         "habilitado", "auditor", "pacto_ambiental", "latitude", "longitude")

perfil(ref)
print("\nfases (formato longo):", fases.count(), "linhas")
fases.groupBy("fase").pivot("resultado").count().orderBy("fase").show(truncate=False)

## 4. Validar antes de gravar

In [ ]:
regras = {
    "224 municípios":        ref.count() == 224,
    "município único":       ref.select("municipio").distinct().count() == ref.count(),
    "código IBGE único":     ref.select("cod_ibge").distinct().count() == ref.count(),
    "toda linha com coord":  ref.filter("latitude IS NULL OR longitude IS NULL").count() == 0,
    "situação preenchida":   ref.filter("situacao IS NULL").count() == 0,
    "selo só quando tem":    ref.filter("tem_selo = true AND selo IS NULL").count() == 0,
    "3 fases x 224":         fases.count() == 672,
    "fase completa":         fases.groupBy("fase").count().filter("count <> 224").count() == 0,
    # `resultado` NULL significaria um rotulo novo na fonte que ninguem mapeou —
    # o painel mostraria a linha em branco em vez de avisar. Melhor falhar aqui.
    "todo resultado mapeado": ref.filter("resultado IS NULL").count() == 0,
    "habilitado sem NULL":   fases.filter("habilitado IS NULL").count() == 0,
    # Nao habilitado (inclui nao postulado) nao pode ter apuracao.
    "sem apuração se inabilitado":
        fases.filter("habilitado = false AND criterios_atendidos IS NOT NULL").count() == 0,
}
for regra, passou in regras.items():
    print(f"  {'OK  ' if passou else 'FALHOU'}  {regra}")

# A regra da edicao: o selo sai do NUMERO DE CRITERIOS, nao da pontuacao.
# Nao vira assert (a fonte manda), mas o que destoa tem de aparecer aqui.
faixa = (F.when(F.col("criterios_atendidos") >= 6, "Selo A")
          .when(F.col("criterios_atendidos") >= 4, "Selo B")
          .when(F.col("criterios_atendidos") == 3, "Selo C")
          .when(F.col("criterios_atendidos") >= 0, "Não elegível"))
fora = fases.filter(F.col("criterios_atendidos").isNotNull() & (faixa != F.col("resultado")))
print(f"\n  municípios fora da regra critérios->selo: {fora.count()}")
fora.select("fase", "municipio", "criterios_atendidos", "resultado", "pontos").show(truncate=False)

# Municipio habilitado e avaliado, mas sem numero de criterios na fonte: nao
# invalida a carga (o resultado veio), mas some do grafico de criterios.
sem_criterio = fases.filter("habilitado = true AND criterios_atendidos IS NULL")
print(f"  habilitados sem critérios na fonte: {sem_criterio.count()}")
sem_criterio.select("fase", "municipio", "resultado", "pontos").show(truncate=False)

assert all(regras.values()), "corrija antes de gravar"

## 5. Gravar as tabelas em `refinamento`

In [ ]:
# Nome com o ano: a edicao 2026 ocupa `semarh_painel`/`semarh_painel_fases`, e as
# duas edicoes convivem no mesmo namespace, cada uma servindo a sua aba do painel.
gravar(ref,   "refinamento.semarh_painel_2025",       modo="substituir")
gravar(fases, "refinamento.semarh_painel_2025_fases", modo="substituir")

## 6. Resumos para o painel

Dois recortes que o dashboard consome direto: municípios **por tipo de selo/situação** e
**por número de critérios atendidos**.

In [ ]:
por_selo = (
    ref.groupBy("resultado")
       .agg(F.count("*").alias("n_municipios"))
       .orderBy("resultado")
)
por_criterios = (
    ref.groupBy("criterios_atendidos")
       .agg(F.count("*").alias("n_municipios"))
       .orderBy("criterios_atendidos")
)
gravar(por_selo,      "refinamento.semarh_painel_2025_por_selo",      modo="substituir")
gravar(por_criterios, "refinamento.semarh_painel_2025_por_criterios", modo="substituir")

## 7. Conferir o resultado

In [ ]:
listar(spark, "refinamento")
print("\nMunicípios por situação (resultado final):")
ref.groupBy("situacao").count().orderBy(F.desc("count")).show(truncate=False)
print("Municípios por tipo de selo (resultado final):")
por_selo.show(truncate=False)
print("Evolução entre as fases:")
(fases.groupBy("resultado").pivot("fase", [1, 2, 3]).count()
      .orderBy("resultado").show(truncate=False))
print("Pontuação e resultado (amostra, maiores pontuações):")
ref.select("municipio", "resultado", "criterios_atendidos", "pontos", "auditor") \
   .orderBy(F.desc("pontos")).show(10, truncate=False)